# Creación de un sistema generativo de texto basado en n-gramas


Como punto de partida para la creación de un sistema generativo de texto es necesario contar con un texto de entrenamiento. En nuestro caso contamos con el archivo "llm/textos.txt" (dentro de la carpeta `llm`) que contiene fragmentos de obras literarias que están en el dominio público, incluyendo a Cervantes, Rubén Darío, Emilio Castelar, Conan Doyle, Félix María Samaniego o Armando Palacio Valdés. Son fragmentos de sus obras descargados del [Proyecto Gutenberg](https://www.gutenberg.org). Aquí vemos algunas líneas del texto:

In [ ]:
# URL absoluta del archivo en GitHub (raw)
ruta_archivo = 'https://raw.githubusercontent.com/MireiaUB/jugandoIA/main/llm/textos.txt'

import requests

# Leer el archivo desde GitHub
respuesta = requests.get(ruta_archivo, timeout=30)
respuesta.raise_for_status()
texto = respuesta.text

# Imprimir las primeras palabras
print(texto[245068:245958])


la izquierda, el mundo de la historia, los muros donde se grabaron
mil victorias; la Vía Sacra, por do entraban los triunfadores; el
Foro, en que se congregaba el pueblo; los arcos bajo los cuales han
pasado veinte siglos sin desgastarlos; las termas regaladas, en
cuyos dibujos todavía se han ceñido su corona las artes modernas;
el Coliseo, que es una montaña esculpida por gigantescos cinceles;
el Quirinal, donde se alzan las mayores estatuas salvadas de las
catástrofes de Grecia; el Capitolio, cabeza, cerebro de la tierra;
y á la vista de tantas maravillas, al recuerdo de tantas grandezas,
á la contemplacion de tantos monumentos engarzados en bosques de
cipreses, que parecen una corona fúnebre sobre la ciudad, colocada
por un genio invisible; cuando las campanas que tocan á la oracion
os envian sus tañidos melancólicos, que os parecen la voz de los
mártires saliendo de las ca


Este texto lo podemos dividir en bigramas, que son grupos de dos palabras:
Mostramos solo unos pocos.


In [149]:
def generar_bigramas(texto):
    """Genera bigramas a partir de un texto.

    Parámetros:
    texto (str): Texto de entrada.

    Retorna:
    list[tuple[str, str]]: Lista de pares de palabras consecutivas.
    """
    # Dividir el texto en palabras
    palabras = texto.split()

    # Generar bigramas
    bigramas = [(palabras[i], palabras[i + 1]) for i in range(len(palabras) - 1)]

    return bigramas

bigramas = generar_bigramas(texto)
print(bigramas[5:40])

[('Mancha,', 'de'), ('de', 'cuyo'), ('cuyo', 'nombre'), ('nombre', 'no'), ('no', 'quiero'), ('quiero', 'acordarme,'), ('acordarme,', 'no'), ('no', 'ha'), ('ha', 'mucho'), ('mucho', 'tiempo'), ('tiempo', 'que'), ('que', 'vivía'), ('vivía', 'un'), ('un', 'hidalgo'), ('hidalgo', 'de'), ('de', 'los'), ('los', 'de'), ('de', 'lanza'), ('lanza', 'en'), ('en', 'astillero,'), ('astillero,', 'adarga'), ('adarga', 'antigua,'), ('antigua,', 'rocín'), ('rocín', 'flaco'), ('flaco', 'y'), ('y', 'galgo'), ('galgo', 'corredor.'), ('corredor.', 'Una'), ('Una', 'olla'), ('olla', 'de'), ('de', 'algo'), ('algo', 'más'), ('más', 'vaca'), ('vaca', 'que'), ('que', 'carnero,')]


También podemos crear n-gramas, trigramas, 4-gramas...:
Mostramos solo unos pocos

In [150]:
def generar_ngramas(texto, n):
    """Genera n-gramas de tamaño n a partir de un texto.

    Parámetros:
    texto (str): Texto de entrada.
    n (int): Tamaño del n-grama.

    Retorna:
    list[tuple[str, ...]]: Lista de n-gramas consecutivos.
    """
    # Dividir el texto en palabras
    palabras = texto.split()

    # Verificar que n es válido
    if n < 1 or n > len(palabras):
        raise ValueError("El tamaño de n-grama debe ser mayor que 0 y menor o igual al número de palabras en el texto")

    # Generar n-gramas
    ngramas = [tuple(palabras[i:i + n]) for i in range(len(palabras) - n + 1)]

    return ngramas

# Ejemplo de uso
n = 3  # Cambia este valor por el tamaño deseado de n-gramas
ngramas = generar_ngramas(texto, n)
print(ngramas[5:80])


[('Mancha,', 'de', 'cuyo'), ('de', 'cuyo', 'nombre'), ('cuyo', 'nombre', 'no'), ('nombre', 'no', 'quiero'), ('no', 'quiero', 'acordarme,'), ('quiero', 'acordarme,', 'no'), ('acordarme,', 'no', 'ha'), ('no', 'ha', 'mucho'), ('ha', 'mucho', 'tiempo'), ('mucho', 'tiempo', 'que'), ('tiempo', 'que', 'vivía'), ('que', 'vivía', 'un'), ('vivía', 'un', 'hidalgo'), ('un', 'hidalgo', 'de'), ('hidalgo', 'de', 'los'), ('de', 'los', 'de'), ('los', 'de', 'lanza'), ('de', 'lanza', 'en'), ('lanza', 'en', 'astillero,'), ('en', 'astillero,', 'adarga'), ('astillero,', 'adarga', 'antigua,'), ('adarga', 'antigua,', 'rocín'), ('antigua,', 'rocín', 'flaco'), ('rocín', 'flaco', 'y'), ('flaco', 'y', 'galgo'), ('y', 'galgo', 'corredor.'), ('galgo', 'corredor.', 'Una'), ('corredor.', 'Una', 'olla'), ('Una', 'olla', 'de'), ('olla', 'de', 'algo'), ('de', 'algo', 'más'), ('algo', 'más', 'vaca'), ('más', 'vaca', 'que'), ('vaca', 'que', 'carnero,'), ('que', 'carnero,', 'salpicón'), ('carnero,', 'salpicón', 'las'), ('s

Así que podemos crear una lista en la que tengamos los n-gramas de diferentes tamaños (n=1, n=2...) acompañados por el número de veces que aparecen en el texto:

In [151]:
def contar_ngramas(lista_ngramas):
    """Cuenta cuántas veces aparece cada n-grama.

    Parámetros:
    lista_ngramas (list[tuple]): N-gramas generados para un tamaño n.

    Retorna:
    list[tuple[tuple, int]]: Pares (ngrama, frecuencia).
    """
    frecuencias_por_ngrama = {}
    for ngrama in lista_ngramas:
        if ngrama in frecuencias_por_ngrama:
            frecuencias_por_ngrama[ngrama] += 1
        else:
            frecuencias_por_ngrama[ngrama] = 1

    pares_ngrama_frecuencia = [
        (ngrama, frecuencia) for ngrama, frecuencia in frecuencias_por_ngrama.items()
    ]
    return pares_ngrama_frecuencia

def generar_lista_ngramas_ordenada_por_conteo(texto_corpus):
    """Genera, para n de 1 a 5, listas de n-gramas ordenadas por frecuencia.

    Parámetros:
    texto_corpus (str): Texto completo del corpus.

    Retorna:
    list[list[tuple[tuple, int]]]: Una lista por cada n, ya ordenada
    de mayor a menor frecuencia.
    """
    lista_palabras = texto_corpus.split()
    if len(lista_palabras) < 5:
        raise ValueError("El texto del corpus debe tener al menos 5 palabras")

    listas_ordenadas_por_tamano = []
    for tamano_n in range(1, 6):  # n = 1, 2, 3, 4, 5
        ngramas_generados = generar_ngramas(texto_corpus, tamano_n)
        ngramas_con_frecuencia = contar_ngramas(ngramas_generados)
        ngramas_ordenados = sorted(
            ngramas_con_frecuencia, key=lambda par: par[1], reverse=True
        )
        listas_ordenadas_por_tamano.append(ngramas_ordenados)
    return listas_ordenadas_por_tamano

# Usar el texto de entrenamiento, evitando conflictos con la variable `texto`
texto_entrenamiento = respuesta.text if 'respuesta' in globals() else texto
lista_ngramas_ordenados_por_conteo = generar_lista_ngramas_ordenada_por_conteo(texto_entrenamiento)

# Alias para compatibilidad con celdas que aún usan el nombre anterior
lista_ngramas_con_conteo = lista_ngramas_ordenados_por_conteo

for indice_n, lista_ordenada in enumerate(lista_ngramas_ordenados_por_conteo, start=1):
    print(f"N-gramas ordenados por conteo para n={indice_n}:\n{lista_ordenada[:10]}\n")

N-gramas ordenados por conteo para n=1:
[(('de',), 11219), (('la',), 7484), (('y',), 6143), (('el',), 5349), (('en',), 4843), (('que',), 4783), (('los',), 3802), (('las',), 2916), (('se',), 2412), (('á',), 2243)]

N-gramas ordenados por conteo para n=2:
[(('de', 'la'), 1610), (('de', 'los'), 813), (('en', 'el'), 727), (('en', 'la'), 670), (('de', 'las'), 620), (('*', '*'), 456), (('á', 'la'), 433), (('y', 'el'), 412), (('de', 'su'), 383), (('que', 'se'), 337)]

N-gramas ordenados por conteo para n=3:
[(('*', '*', '*'), 342), (('Véase', 'la', 'nota'), 40), (('_m.', '&', 'f._,'), 39), (('y', 'de', 'la'), 38), (('y', 'en', 'el'), 38), (('de', 'todos', 'los'), 33), (('en', 'vez', 'de'), 32), (('de', 'la', 'Edad'), 32), (('--a=,', '_m.', '&'), 32), (('uno', 'de', 'los'), 29)]

N-gramas ordenados por conteo para n=4:
[(('*', '*', '*', '*'), 228), (('--a=,', '_m.', '&', 'f._,'), 32), (('la', 'nota', '2,', 'pág.'), 23), (('en', 'el', 'fondo', 'de'), 18), (('la', 'nota', '1,', 'pág.'), 17), (('

Esto nos puede servir para, a partir de un texto, generar la siguiente palabra más probable. Así, si partimos del texto "de vez en cuando", podemos mirar la lista de 5-gramas y comprobar si hay alguno en los que las cuatro primeras palabras sean "de" "vez" "en" "cuando". Si es así se devuelve la quinta palabra del 5-grama. Si hay varios 5-gramas que cumplan la condición se devuelve la palabra de aquel que ocurre más veces en el texto. Por tanto, la palabra más probable tras "de vez en cuando" es:

In [153]:
def buscar_quinta_palabra(lista_5gramas_ordenados_por_conteo, texto):
    """Busca la quinta palabra más probable para un contexto de 4 palabras.

    Nota: la lista de 5-gramas debe estar ordenada por frecuencia descendente,
    por lo que la primera coincidencia ya es la más frecuente.

    Parámetros:
    lista_5gramas_ordenados_por_conteo (list[tuple[tuple, int]]): 5-gramas con su frecuencia.
    texto (str): Contexto de exactamente 4 palabras.

    Retorna:
    str | None: Quinta palabra más probable para ese contexto.
    """
    palabras_contexto = texto.split()
    if len(palabras_contexto) != 4:
        raise ValueError("El texto debe contener exactamente 4 palabras")

    contexto = tuple(palabras_contexto)

    # Como la lista ya está ordenada por frecuencia, devolvemos la primera coincidencia.
    for ngrama, _frecuencia in lista_5gramas_ordenados_por_conteo:
        if ngrama[:4] == contexto:
            return ngrama[4]

    return None

texto = "de vez en cuando"
quinta_palabra = buscar_quinta_palabra(lista_ngramas_ordenados_por_conteo[4], texto)
print(quinta_palabra)

el


Así que esto mismo podemos hacerlo con textos de diferentes tamaños. Miramos en el n-grama con mayor valor de n (en nuestro caso, 5). Si en esa lista no encontramos ninguna coincidencia, miramos en los 4-gramas. Si tampoco hay coincidencia, pasamos a los 3-gramas. Y así hasta los 1-grama. Si tampoco hay coincidencia se devuelve al azar una palabra de los 1-gramas.

Así, por ejemplo, vemos a continuación cuál es la siguiente palabra más probable a partir del texto: "de la casa de"

In [154]:
import random

def siguiente_palabra(listas_ngramas_con_conteo, texto):
    """Predice la siguiente palabra usando coincidencias de n-gramas.

    Parámetros:
    listas_ngramas_con_conteo (list): Listas de n-gramas con conteo para n=1..5.
    texto (str): Texto de contexto.

    Retorna:
    str: Siguiente palabra más probable.
    """
    palabras = texto.split()

    # Iterar desde 5-gramas hasta 2-gramas
    for i in range(4, 0, -1):
        ngramas_con_conteo = listas_ngramas_con_conteo[i]
        subconjunto_palabras = tuple(palabras[-i:])

        # Buscar el n-grama con el mayor conteo que coincida
        palabra_candidata, conteo_maximo = None, 0
        for ngrama, conteo in ngramas_con_conteo:
            if ngrama[:i] == subconjunto_palabras and conteo > conteo_maximo:
                palabra_candidata = ngrama[i]
                conteo_maximo = conteo

        if palabra_candidata:
            return palabra_candidata

    # Si no se encuentra ninguna coincidencia, escoger al azar de los 2-gramas
    return random.choice(listas_ngramas_con_conteo[0])[0][0]

texto = "de la casa de"
palabra_siguiente = siguiente_palabra(lista_ngramas_con_conteo, texto)
print(palabra_siguiente)

Meira


Y ahora la palabra más probable para el texto "en el fondo de"

In [155]:
texto = "en el fondo de"
palabra_siguiente = siguiente_palabra(lista_ngramas_con_conteo, texto)
print(palabra_siguiente)



su


Y, de este modo, podríamos partir de un texto y generar otro nuevo añadiendo cada vez la siguiente palabra más probable. Así, si partimos del texto "En medio de" y pedimos que continúe escribiendo otras 15 palabras obtenemos lo siguiente:

Observar que el texto generado no aparece exactamente en ninguno de los originales introducidos.

In [156]:
def generar_texto_autoregresivo(listas_ngramas_con_conteo, texto_original, n):
    """Genera texto añadiendo n palabras predichas.

    Parámetros:
    listas_ngramas_con_conteo (list): Listas de n-gramas con conteo para n=1..5.
    texto_original (str): Texto semilla.
    n (int): Número de palabras nuevas a generar.

    Retorna:
    str: Texto generado final.
    """
    texto = texto_original
    for _ in range(n):
        nueva_palabra = siguiente_palabra(listas_ngramas_con_conteo, texto)
        texto += ' ' + nueva_palabra
        print(texto)
    return texto


In [157]:
texto_original = "El gozo de la Naturaleza"
nuevas_palabras = 15

In [158]:
texto_generado = generar_texto_autoregresivo(lista_ngramas_con_conteo, texto_original, nuevas_palabras)

El gozo de la Naturaleza raya
El gozo de la Naturaleza raya en
El gozo de la Naturaleza raya en usted
El gozo de la Naturaleza raya en usted en
El gozo de la Naturaleza raya en usted en adoración
El gozo de la Naturaleza raya en usted en adoración panteística.
El gozo de la Naturaleza raya en usted en adoración panteística. Hay
El gozo de la Naturaleza raya en usted en adoración panteística. Hay en
El gozo de la Naturaleza raya en usted en adoración panteística. Hay en las
El gozo de la Naturaleza raya en usted en adoración panteística. Hay en las cuatro
El gozo de la Naturaleza raya en usted en adoración panteística. Hay en las cuatro composiciones
El gozo de la Naturaleza raya en usted en adoración panteística. Hay en las cuatro composiciones a
El gozo de la Naturaleza raya en usted en adoración panteística. Hay en las cuatro composiciones a (_a_
El gozo de la Naturaleza raya en usted en adoración panteística. Hay en las cuatro composiciones a (_a_ o
El gozo de la Naturaleza raya en 

Esto mismo es lo que hacen los grandes modelos de lenguaje como GPT3 o Bloom, pero en su caso la n de los n-grama es muchísimo más grande, ya que utilizan una arquitectura de redes neuronales llamada transformer que puede trabajar con contextos de miles de palabras. Y es precisamente esa posibilidad de considerar contextos tan grandes lo que los hace tan precisos al continuar textos. Además, el conjunto de textos de entrenamiento es también inmenso, por lo que pueden continuar textos de muchas temáticas diferentes.

# Créditos

Este cuaderno es una adaptación del [cuaderno](https://colab.research.google.com/drive/12WY2pQXu_FMCbGN2uV1WdlH00yWJhcp-?usp=sharing) elaborado por [Jesús Moreno](http://jemole.me), que a su vez está inspirado en el trabajo de Jens Mönig para la creación de [SnapGPT](https://www.youtube.com/watch?v=32bbKGggdIA). El vídeo enlazado, en el que describe su desarrollo, es muy interesante y recomendable.